In [31]:
## Gabriel Lagares Gontijo ra: 22252596


# 🎵 Sistema de Recomendação Musical com FastAPI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabriellgontijo/python/blob/main/Sistema_Recomendacao_Musical.ipynb)

**Atividade Prática**: sistema de recomendação musical com **5 APIs** em FastAPI, sobre o dataset
*Top 50 do Spotify (2010–2019)* (`top50MusicFrom2010-2019.csv`) disponibilizado pelo professor.

| # | Método | Endpoint | Abordagem |
|---|--------|----------|-----------|
| 1 | GET  | `/recommendations/content-based/{song_title}` | Baseada em conteúdo (similaridade do cosseno) |
| 2 | POST | `/recommendations/genre-artist` | Mesmo gênero ou artista, ordenado por popularidade |
| 3 | GET  | `/recommendations/collaborative/{user_id}` | Filtro colaborativo (contagem de co-ocorrências) |
| 4 | POST | `/recommendations/hybrid` | Híbrida (conteúdo + colaborativo) |
| 5 | GET  | `/recommendations/popular` | Mais populares, filtrando por ano/gênero |

**Como executar:** menu **Ambiente de execução → Executar tudo** (`Ctrl+F9`). O notebook:

1. instala as dependências;
2. baixa o CSV (se não conseguir, pede o upload do arquivo);
3. grava o código da API em `main.py`;
4. roda os testes automatizados (`pytest`);
5. sobe a API com `uvicorn` e chama cada endpoint, mostrando requisição e resposta;
6. gera um link para a documentação interativa (Swagger) da API.

## 1. Instalação das dependências

In [1]:
%pip install -q fastapi uvicorn httpx pytest pandas scikit-learn requests

## 2. Base de dados

Cada linha é uma música do Top 50 de um ano entre 2010 e 2019, com título, artista, gênero, ano,
características musicais (BPM, energia, danceabilidade, volume, ...) e popularidade.

A célula abaixo procura o CSV na pasta atual. Se não o encontrar, baixa a cópia do GitHub e, se
isso falhar, pede o upload do arquivo do professor.

In [2]:
import glob
import os
import urllib.request

CSV_NAME = "top50MusicFrom2010-2019.csv"
CSV_URL = "https://raw.githubusercontent.com/gabriellgontijo/python/cab5baa1fd7653b05ed8104b6df9b174b91f73ac/top50MusicFrom2010-2019%281%29.csv"

encontrados = sorted(glob.glob("top50*.csv"))
if encontrados:
    CSV_PATH = encontrados[0]
    print("CSV encontrado:", CSV_PATH)
else:
    try:
        urllib.request.urlretrieve(CSV_URL, CSV_NAME)
        print("CSV baixado do GitHub:", CSV_NAME)
    except Exception as erro:
        print("Não foi possível baixar o CSV:", erro)
        from google.colab import files  # fora do Colab: coloque o CSV na pasta do notebook
        print("Selecione o arquivo top50MusicFrom2010-2019.csv:")
        enviado = next(iter(files.upload()))
        os.replace(enviado, CSV_NAME)
    CSV_PATH = CSV_NAME

CSV baixado do GitHub: top50MusicFrom2010-2019.csv


In [3]:
import pandas as pd

pd.set_option("display.max_colwidth", None)

bruto = pd.read_csv(CSV_PATH)
print(f"{bruto.shape[0]} músicas x {bruto.shape[1]} colunas\n")
print("Colunas originais do CSV:")
for coluna in bruto.columns:
    print("  -", coluna)
bruto.head()

603 músicas x 14 colunas

Colunas originais do CSV:
  - title
  - artist
  - the genre of the track
  - year
  - Beats.Per.Minute -The tempo of the song
  - Energy- The energy of a song - the higher the value, the more energtic
  - Danceability - The higher the value, the easier it is to dance to this song
  - Loudness/dB - The higher the value, the louder the song
  - Liveness - The higher the value, the more likely the song is a live recording
  - Valence - The higher the value, the more positive mood for the song
  - Length - The duration of the song
  - Acousticness - The higher the value the more acoustic the song is
  - Speechiness - The higher the value the more spoken word the song contains
  - Popularity- The higher the value the more popular the song is


,title,artist,the genre of the track,year,Beats.Per.Minute -The tempo of the song,"Energy- The energy of a song - the higher the value, the more energtic","Danceability - The higher the value, the easier it is to dance to this song","Loudness/dB - The higher the value, the louder the song","Liveness - The higher the value, the more likely the song is a live recording","Valence - The higher the value, the more positive mood for the song",Length - The duration of the song,Acousticness - The higher the value the more acoustic the song is,Speechiness - The higher the value the more spoken word the song contains,Popularity- The higher the value the more popular the song is
0,"Hey, Soul Sister",Train,neo mellow,2010,97,89,67,-4,8,80,217,19,4,83
1,Love The Way You Lie,Eminem,detroit hip hop,2010,87,93,75,-5,52,64,263,24,23,82
2,TiK ToK,Kesha,dance pop,2010,120,84,76,-3,29,71,200,10,14,80
3,Bad Romance,Lady Gaga,dance pop,2010,119,92,70,-4,8,71,295,0,4,79
4,Just the Way You Are,Bruno Mars,pop,2010,109,84,64,-5,9,43,221,2,4,78


## 3. Código da API (`main.py`)

O código segue a **estrutura do código base** do enunciado: `FastAPI`, modelos Pydantic
`GenreArtistRequest` e `HybridRequest`, lista `features`, `MinMaxScaler` e `cosine_similarity`.

**Pré-processamento dos dados**

- **Nomes de coluna**: os nomes longos do CSV (ex.: `Energy- The energy of a song - ...`) viram
  os nomes do código base (`Beats.Per.Minute`, `Energy`, `Danceability`, `Loudness/dB`, ...).
- **Caracteres corrompidos**: 10 títulos vieram com acentos quebrados no CSV (ex.: `Se駉rita`) e
  são corrigidos (`Señorita`).
- **Registro inválido removido**: `Million Years Ago` (Adele) tem BPM 0 e todas as
  características zeradas. Ficam 602 músicas.
- **Normalização**: `MinMaxScaler` nas 10 características e matriz de similaridade do cosseno.
  As respostas mostram os valores originais.
- **Repetidas**: 16 músicas aparecem duas vezes no CSV (ex.: `Love Yourself` em 2015 e 2016). Nas
  recomendações cada uma aparece uma única vez.

**Decisões de implementação**

- **Busca de títulos**: não diferencia maiúsculas/minúsculas nem acentos, e aceita o título sem
  complemento (`Titanium` encontra `Titanium (feat. Sia)`). Se a música não existir, a resposta 404
  sugere títulos parecidos.
- **`weights` como JSON na URL**: na assinatura do código base
  (`weights: Optional[Dict[str, float]]`), o FastAPI trata um `Dict` como **corpo** da requisição.
  Em um GET, o Swagger não envia corpo e `?weights=` na URL é ignorado. Por isso o parâmetro é
  recebido como texto JSON (ex.: `{"Energy": 2, "Danceability": 1.5}`) e convertido em dicionário.
- **Filtro colaborativo**: dicionário `user_likes` com 15 usuários fictícios (`user1` a `user15`)
  e as músicas que cada um curtiu. A matriz `co_occurrence[A][B]` conta quantos usuários gostaram
  de A e de B.
- **Híbrida**: `hybrid_score = content_weight × content_score + collab_weight × collab_score`,
  com os dois scores normalizados entre 0 e 1 e os pesos normalizados para somar 1.
- **Documentação**: cada endpoint tem descrição e exemplos de requisição/resposta, visíveis no
  Swagger (`/docs`).

In [4]:
%%writefile main.py
"""
Sistema de Recomendação Musical com FastAPI
===========================================

Atividade Prática: 5 APIs de recomendação sobre o dataset
"top50MusicFrom2010-2019.csv" (Top 50 do Spotify de 2010 a 2019).

    1. GET  /recommendations/content-based/{song_title}  -> similaridade por características
    2. POST /recommendations/genre-artist                -> mesmo gênero ou artista
    3. GET  /recommendations/collaborative/{user_id}     -> filtro colaborativo (co-ocorrências)
    4. POST /recommendations/hybrid                      -> conteúdo + colaborativo
    5. GET  /recommendations/popular                     -> mais populares por ano/gênero

Como executar localmente:
    pip install -r requirements.txt
    uvicorn main:app --reload
    Documentação interativa (Swagger): http://127.0.0.1:8000/docs
"""

import difflib
import json
import math
import os
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel, ConfigDict, Field
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

app = FastAPI(
    title="Sistema de Recomendação Musical",
    description=(
        "Atividade Prática: 5 APIs de recomendação musical (conteúdo, gênero/artista, "
        "filtro colaborativo, híbrida e popularidade) usando o dataset Top 50 do Spotify "
        "de 2010 a 2019.\n\n"
        "Usuários fictícios do filtro colaborativo: `user1` a `user15` (veja `GET /users`)."
    ),
    version="1.0.0",
)


# =============================================================================
# Carregar dados
# =============================================================================

# O CSV do professor usa nomes de coluna longos e descritivos
# (ex.: "Energy- The energy of a song - the higher the value, ...").
# Eles são renomeados para os nomes curtos usados no código base do enunciado.
COLUMN_PREFIXES = {
    "title": "title",
    "artist": "artist",
    "the genre of the track": "genre",
    "year": "year",
    "beats.per.minute": "Beats.Per.Minute",
    "energy": "Energy",
    "danceability": "Danceability",
    "loudness/db": "Loudness/dB",
    "liveness": "Liveness",
    "valence": "Valence",
    "length": "Length",
    "acousticness": "Acousticness",
    "speechiness": "Speechiness",
    "popularity": "Popularity",
}

features = ['Beats.Per.Minute', 'Energy', 'Danceability', 'Loudness/dB',
            'Liveness', 'Valence', 'Length', 'Acousticness', 'Speechiness', 'Popularity']

# Caracteres chineses indicam texto corrompido ("mojibake"): o arquivo original em
# Windows-1252 foi lido como GBK antes de ser salvo em UTF-8 (ex.: "Se駉rita").
_MOJIBAKE = re.compile(r"[㐀-鿿豈-﫿]")
_QUOTES = str.maketrans({"‘": "'", "’": "'", "“": '"', "”": '"'})


def find_csv() -> Path:
    """Procura o CSV na pasta deste arquivo ou na pasta atual (ex.: /content no Colab)."""
    if os.environ.get("MUSIC_CSV_PATH"):
        return Path(os.environ["MUSIC_CSV_PATH"])
    for folder in (Path(__file__).resolve().parent, Path.cwd()):
        found = sorted(folder.glob("top50*.csv"))
        if found:
            return found[0]
    raise FileNotFoundError(
        "Arquivo 'top50MusicFrom2010-2019.csv' não encontrado. "
        "Coloque o CSV na mesma pasta do main.py."
    )


def fix_mojibake(text: str) -> str:
    """Desfaz a corrupção de caracteres: 'Se駉rita' -> 'Señorita'."""
    if _MOJIBAKE.search(text):
        try:
            return text.encode("gbk").decode("cp1252")
        except (UnicodeEncodeError, UnicodeDecodeError):
            return text
    return text


def normalize(text) -> str:
    """Chave de busca: minúsculas, sem acentos, aspas simples e espaços padronizados."""
    text = unicodedata.normalize("NFKD", str(text).translate(_QUOTES))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return " ".join(text.lower().split())


def load_dataset(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path, encoding="utf-8")

    rename = {}
    for column in raw.columns:
        key = column.strip().lower()
        for prefix, name in COLUMN_PREFIXES.items():
            if key.startswith(prefix):
                rename[column] = name
                break
    data = raw.rename(columns=rename)
    required = ["title", "artist", "genre", "year"] + features
    missing = [column for column in required if column not in data.columns]
    if missing:
        raise ValueError(f"Colunas não encontradas no CSV: {missing}")
    data = data[required].copy()

    data["original_title"] = data["title"].astype(str).str.strip()
    for column in ["title", "artist", "genre"]:
        data[column] = data[column].astype(str).str.strip().map(fix_mojibake)

    # Registro inválido: "Million Years Ago" (Adele) tem BPM 0 e todas as
    # características zeradas. Além de não ser uma música "real" para comparação,
    # ele distorceria a normalização (Loudness de -60 dB).
    data = data[data["Beats.Per.Minute"] > 0].reset_index(drop=True)

    # Chaves normalizadas usadas nas buscas (sem diferenciar maiúsculas/acentos)
    data["title_key"] = data["title"].map(normalize)
    data["original_title_key"] = data["original_title"].map(normalize)
    # Título "base", sem sufixos como "(feat. ...)" ou "- Remix": "Titanium" encontra "Titanium (feat. Sia)"
    data["base_title_key"] = data["title_key"].map(lambda t: re.split(r"\s+[-(\[]", t, maxsplit=1)[0])
    data["artist_key"] = data["artist"].map(normalize)
    data["genre_key"] = data["genre"].map(normalize)
    data["song_key"] = data["title_key"] + " | " + data["artist_key"]
    return data


df = load_dataset(find_csv())

# Pré-processamento: as características são normalizadas entre 0 e 1 (MinMaxScaler)
# em uma matriz separada, para que as respostas mostrem os valores originais.
scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(df[features])

# Modelo de similaridade (cosseno entre todas as músicas)
similarity_matrix = cosine_similarity(features_scaled)

# 16 músicas aparecem duas vezes no CSV (ex.: "Love Yourself" em 2015 e 2016).
# Nas recomendações cada música aparece uma única vez (a versão mais popular).
songs = df.sort_values("Popularity", ascending=False, kind="mergesort").drop_duplicates("song_key")
canonical_index = dict(zip(songs["song_key"], songs.index))


# =============================================================================
# Dados fictícios para o filtro colaborativo
# =============================================================================

# "Usuários que gostaram disso também gostaram daquilo": cada usuário fictício tem
# uma lista de músicas curtidas. Os gostos formam grupos (pop dançante, pop/funk,
# EDM, pop canadense, baladas, latino) com algumas músicas em comum entre grupos.
user_likes: Dict[str, List[str]] = {
    # Pop dançante (Katy Perry, Lady Gaga, Rihanna, Kesha)
    "user1": ["Roar", "Dark Horse", "Bad Romance", "TiK ToK", "Only Girl (In The World)", "Telephone"],
    "user2": ["Roar", "Bad Romance", "Telephone", "Die Young", "Part Of Me", "Diamonds", "We Found Love"],
    "user3": ["Dark Horse", "TiK ToK", "Die Young", "Starships", "Problem", "Side To Side",
              "Titanium (feat. Sia)"],
    # Pop/funk (Bruno Mars, Maroon 5)
    "user4": ["Just the Way You Are", "Locked Out of Heaven", "Uptown Funk", "Sugar", "Payphone",
              "Treasure", "That's What I Like"],
    "user5": ["Uptown Funk", "That's What I Like", "24K Magic", "Sugar", "Animals", "Marry You",
              'Happy - From "Despicable Me 2"'],
    # EDM
    "user6": ["Wake Me Up", "Summer", "Titanium (feat. Sia)", "Clarity", "Don't Let Me Down", "Closer",
              "Hey Brother"],
    "user7": ["Wake Me Up", "Titanium (feat. Sia)", "This Is What You Came For", "Closer",
              "Something Just Like This", "Stay"],
    "user8": ["Summer", "How Deep Is Your Love", "One Kiss (with Dua Lipa)", "New Rules", "Silence",
              "Happier", "Don't Let Me Down"],
    # Pop canadense (Justin Bieber, Shawn Mendes, The Weeknd)
    "user9": ["Love Yourself", "Sorry", "What Do You Mean?", "Stitches", "Treat You Better", "Starboy",
              "The Hills"],
    "user10": ["Love Yourself", "Sorry", "Stitches", "Mercy", "There's Nothing Holdin' Me Back", "Baby",
               "Boyfriend"],
    "user11": ["Starboy", "The Hills", "I Feel It Coming", "Too Good", "Hold On, We're Going Home",
               "Love Yourself"],
    # Baladas / soul (Adele, Ed Sheeran, Sam Smith)
    "user12": ["Someone Like You", "Rolling in the Deep", "Set Fire to the Rain", "Thinking out Loud",
               "Stay With Me", "All of Me"],
    "user13": ["Someone Like You", "Thinking out Loud", "Stay With Me", "I'm Not The Only One", "All I Ask",
               "Let Her Go", "Shape of You"],
    # Latino / festa
    "user14": ["Despacito - Remix", "Havana (feat. Young Thug)", "Give Me Everything", "On The Floor",
               "Taki Taki (feat. Selena Gomez, Ozuna & Cardi B)", "International Love"],
    "user15": ["Despacito - Remix", "Havana (feat. Young Thug)", "Give Me Everything",
               "Hey Mama (feat. Nicki Minaj, Bebe Rexha & Afrojack)", "Shape of You", "Cheap Thrills"],
}


# =============================================================================
# Funções auxiliares
# =============================================================================

def match_title(title: str, artist: Optional[str] = None) -> pd.DataFrame:
    """Linhas do dataset cujo título corresponde à busca (exato ou título base)."""
    pool = df
    if artist and artist.strip():
        pool = pool[pool["artist_key"] == normalize(artist)]
    key = normalize(title)
    matches = pool[(pool["title_key"] == key) | (pool["original_title_key"] == key)]
    if matches.empty:
        matches = pool[pool["base_title_key"] == key]
    return matches


def suggest(query: str, options, n: int = 5) -> List[str]:
    """Sugestões de nomes parecidos, para mensagens de erro mais úteis."""
    by_key = {}
    for option in options:
        by_key.setdefault(normalize(option), option)
    key = normalize(query)
    found = difflib.get_close_matches(key, list(by_key), n=n, cutoff=0.5)
    if len(key) >= 3:
        found += [option_key for option_key in by_key if key in option_key]
    return [by_key[option_key] for option_key in list(dict.fromkeys(found))[:n]]


def find_song_index(song_title: str, artist: Optional[str] = None) -> int:
    """Índice da música buscada. Títulos repetidos: usa a versão mais popular."""
    matches = match_title(song_title, artist)
    if matches.empty:
        message = f"Música '{song_title}' não encontrada"
        message += f" para o artista '{artist}'." if artist else "."
        raise HTTPException(status_code=404, detail={
            "message": message,
            "suggestions": suggest(song_title, df["title"]),
        })
    return int(matches.sort_values("Popularity", ascending=False, kind="mergesort").index[0])


def resolve_user(user_id: str) -> str:
    """Aceita 'user1', 'USER1' ou apenas '1'."""
    key = normalize(user_id)
    if key.isdigit():
        key = f"user{int(key)}"
    for user in user_likes:
        if user.lower() == key:
            return user
    raise HTTPException(status_code=404, detail={
        "message": f"Usuário '{user_id}' não encontrado.",
        "available_users": list(user_likes),
    })


def song_info(row: pd.Series) -> dict:
    return {
        "title": row["title"],
        "artist": row["artist"],
        "genre": row["genre"],
        "year": int(row["year"]),
        "popularity": int(row["Popularity"]),
    }


def song_features(row: pd.Series) -> Dict[str, int]:
    return {feature: int(row[feature]) for feature in features}


def _feature_key(name: str) -> str:
    return re.sub(r"[^a-z0-9]", "", normalize(name))


FEATURE_NAMES = {_feature_key(feature): feature for feature in features}
FEATURE_NAMES.update({
    "bpm": "Beats.Per.Minute", "tempo": "Beats.Per.Minute",
    "loudness": "Loudness/dB", "db": "Loudness/dB", "duration": "Length",
})


def parse_weights(raw: Optional[str]) -> Optional[Dict[str, float]]:
    """Converte o parâmetro weights em um dicionário {característica: peso}.

    Aceita JSON ({"Energy": 2, "Danceability": 1.5}) ou o formato curto
    "Energy:2,Danceability:1.5". Características não informadas ficam com peso 1.
    """
    if raw is None or not raw.strip():
        return None
    text = raw.strip()
    invalid_format = HTTPException(status_code=400, detail={
        "message": 'Formato inválido para weights. Use JSON, ex.: {"Energy": 2, "Danceability": 1.5}, '
                   'ou "Energy:2,Danceability:1.5".',
        "valid_features": features,
    })
    try:
        if text.startswith("{"):
            parsed = json.loads(text)
            if not isinstance(parsed, dict):
                raise invalid_format
            items = list(parsed.items())
        else:
            items = []
            for part in text.split(","):
                name, separator, value = part.partition(":")
                if not separator:
                    name, separator, value = part.partition("=")
                if not separator:
                    raise invalid_format
                items.append((name, value))

        weights = {}
        for name, value in items:
            feature = FEATURE_NAMES.get(_feature_key(name))
            if feature is None:
                raise HTTPException(status_code=400, detail={
                    "message": f"Característica desconhecida em weights: '{name.strip()}'.",
                    "valid_features": features,
                })
            weight = float(value)
            if not math.isfinite(weight) or weight < 0:
                raise HTTPException(status_code=400, detail={
                    "message": f"O peso de '{feature}' deve ser um número maior ou igual a zero.",
                })
            weights[feature] = weight
    except (ValueError, TypeError):
        raise invalid_format

    full_weights = {feature: weights.get(feature, 1.0) for feature in features}
    if sum(full_weights.values()) == 0:
        raise HTTPException(status_code=400, detail={"message": "Pelo menos um peso deve ser maior que zero."})
    return full_weights


def weighted_similarity(index: int, weights: Dict[str, float]) -> np.ndarray:
    """Similaridade do cosseno ponderada: sum(w*x*y) / (sqrt(sum(w*x²)) * sqrt(sum(w*y²)))."""
    scale = np.sqrt([weights[feature] for feature in features])
    weighted = features_scaled * scale
    return cosine_similarity(weighted[index:index + 1], weighted)[0]


# Filtro colaborativo: cada música curtida pelos usuários fictícios é associada a
# uma linha do dataset e contamos quantas vezes cada par de músicas aparece junto.
user_song_keys: Dict[str, List[str]] = {}
for _user, _titles in user_likes.items():
    _keys = []
    for _title in _titles:
        _matches = match_title(_title)
        if _matches.empty:
            raise ValueError(f"Música de {_user} não encontrada no dataset: {_title}")
        _keys.append(df.at[int(_matches.sort_values("Popularity", ascending=False).index[0]), "song_key"])
    user_song_keys[_user] = list(dict.fromkeys(_keys))

# co_occurrence[A][B] = quantos usuários gostaram de A e também de B
co_occurrence: Dict[str, Counter] = defaultdict(Counter)
for _keys in user_song_keys.values():
    for _a in _keys:
        for _b in _keys:
            if _a != _b:
                co_occurrence[_a][_b] += 1


def collaborative_scores(user: str):
    """Score de cada música não curtida = soma das co-ocorrências com as músicas curtidas."""
    liked = set(user_song_keys[user])
    scores: Counter = Counter()
    reasons: Dict[str, List[str]] = defaultdict(list)
    for liked_key in user_song_keys[user]:
        for other_key, count in co_occurrence[liked_key].items():
            if other_key not in liked:
                scores[other_key] += count
                reasons[other_key].append(liked_key)
    return scores, reasons


def title_of(song_key: str) -> str:
    return df.at[canonical_index[song_key], "title"]


def short_info(song_key: str) -> dict:
    row = df.loc[canonical_index[song_key]]
    return {"title": row["title"], "artist": row["artist"]}


# =============================================================================
# Modelos de requisição (corpo dos POST)
# =============================================================================

class GenreArtistRequest(BaseModel):
    genre: Optional[str] = None
    artist: Optional[str] = None
    limit: int = Field(5, ge=1, le=100)

    model_config = ConfigDict(json_schema_extra={
        "examples": [{"genre": "canadian pop", "artist": "Ed Sheeran", "limit": 5}]
    })


class HybridRequest(BaseModel):
    song_title: str
    user_id: str
    content_weight: float = Field(0.7, ge=0)
    collab_weight: float = Field(0.3, ge=0)
    limit: int = Field(5, ge=1, le=100)

    model_config = ConfigDict(json_schema_extra={
        "examples": [{"song_title": "Shape of You", "user_id": "user12", "content_weight": 0.7,
                      "collab_weight": 0.3, "limit": 5}]
    })


# =============================================================================
# Endpoints
# =============================================================================

@app.get("/", tags=["Auxiliares"], summary="Informações da API")
async def root():
    """Resumo do dataset e lista dos endpoints. A documentação interativa fica em `/docs`."""
    return {
        "message": "Sistema de Recomendação Musical",
        "docs": "/docs",
        "dataset": {
            "rows": len(df),
            "unique_songs": len(songs),
            "years": sorted(int(year) for year in df["year"].unique()),
            "genres": int(df["genre"].nunique()),
            "artists": int(df["artist"].nunique()),
        },
        "endpoints": [
            "GET  /recommendations/content-based/{song_title}",
            "POST /recommendations/genre-artist",
            "GET  /recommendations/collaborative/{user_id}",
            "POST /recommendations/hybrid",
            "GET  /recommendations/popular",
        ],
    }


@app.get("/users", tags=["Auxiliares"], summary="Usuários fictícios do filtro colaborativo")
async def list_users():
    """Lista os usuários simulados e as músicas que cada um curtiu."""
    return {
        "total": len(user_likes),
        "users": {user: [short_info(key) for key in keys] for user, keys in user_song_keys.items()},
    }


@app.get("/recommendations/content-based/{song_title:path}", tags=["Recomendações"],
         summary="1. Recomendação baseada em conteúdo")
async def content_based_recommendations(
    song_title: str,
    limit: int = Query(5, ge=1, le=100, description="Número de recomendações a retornar"),
    weights: Optional[str] = Query(
        None,
        description='Pesos por característica (dicionário em JSON). Ex.: {"Energy": 2, "Danceability": 1.5}. '
                    "Características não informadas ficam com peso 1; peso 0 ignora a característica.",
    ),
    artist: Optional[str] = Query(None, description="Opcional: artista, para títulos repetidos (ex.: 'Hello')"),
):
    """
    Retorna as músicas mais parecidas com `song_title` pelas características musicais
    (BPM, energia, danceabilidade, volume, ...), usando **similaridade do cosseno**
    sobre as características normalizadas (MinMaxScaler).

    - A busca do título não diferencia maiúsculas/minúsculas nem acentos, e aceita o
      título sem o complemento (ex.: `Titanium` encontra `Titanium (feat. Sia)`).
    - `weights` é um dicionário `{característica: peso}` enviado como JSON na query string.
      Características válidas: `Beats.Per.Minute`, `Energy`, `Danceability`, `Loudness/dB`,
      `Liveness`, `Valence`, `Length`, `Acousticness`, `Speechiness`, `Popularity`.

    **Exemplo de requisição**

        GET /recommendations/content-based/Shape%20of%20You?limit=2&weights={"Energy":2,"Danceability":2}

    **Exemplo de resposta** (`features` resumido)

        {
          "song": {"title": "Shape of You", "artist": "Ed Sheeran", "genre": "pop", "year": 2017,
                   "popularity": 87, "features": {"Beats.Per.Minute": 96, "Energy": 65, "...": "..."}},
          "weights": {"Beats.Per.Minute": 1.0, "Energy": 2.0, "Danceability": 2.0, "...": 1.0},
          "total": 2,
          "recommendations": [
            {"title": "Closer", "artist": "The Chainsmokers", "genre": "electropop", "year": 2017,
             "popularity": 86, "similarity": 0.9901, "features": {"Beats.Per.Minute": 95, "...": "..."}},
            {"title": "Sing", "artist": "Ed Sheeran", "genre": "pop", "year": 2015,
             "popularity": 71, "similarity": 0.9872, "features": {"Beats.Per.Minute": 120, "...": "..."}}
          ]
        }
    """
    index = find_song_index(song_title, artist)
    parsed_weights = parse_weights(weights)
    similarities = similarity_matrix[index] if parsed_weights is None else weighted_similarity(index, parsed_weights)

    seed = df.loc[index]
    candidates = songs[songs["song_key"] != seed["song_key"]].copy()
    candidates["similarity"] = similarities[candidates.index]
    top = candidates.sort_values(["similarity", "Popularity"], ascending=False, kind="mergesort").head(limit)

    return {
        "song": {**song_info(seed), "features": song_features(seed)},
        "weights": parsed_weights or {feature: 1.0 for feature in features},
        "total": len(top),
        "recommendations": [
            {**song_info(row), "similarity": round(float(row["similarity"]), 4), "features": song_features(row)}
            for _, row in top.iterrows()
        ],
    }


@app.post("/recommendations/genre-artist", tags=["Recomendações"],
          summary="2. Recomendação por gênero/artista")
async def genre_artist_recommendations(request: GenreArtistRequest):
    """
    Retorna músicas do **mesmo gênero OU do mesmo artista** informados, ordenadas por
    popularidade (maior primeiro). Informe `genre`, `artist` ou os dois. A comparação não
    diferencia maiúsculas/minúsculas nem acentos. O campo `match` indica o motivo da
    recomendação: `genre`, `artist` ou `genre+artist`.

    **Exemplo de requisição**

        POST /recommendations/genre-artist
        {"genre": "canadian pop", "artist": "Ed Sheeran", "limit": 3}

    **Exemplo de resposta**

        {
          "genre": "canadian pop",
          "artist": "Ed Sheeran",
          "total_matches": 42,
          "total": 3,
          "recommendations": [
            {"title": "Señorita", "artist": "Shawn Mendes", "genre": "canadian pop", "year": 2019,
             "popularity": 95, "match": "genre"},
            {"title": "South of the Border (feat. Camila Cabello & Cardi B)", "artist": "Ed Sheeran",
             "genre": "pop", "year": 2019, "popularity": 92, "match": "artist"},
            {"title": "Shape of You", "artist": "Ed Sheeran", "genre": "pop", "year": 2017,
             "popularity": 87, "match": "artist"}
          ]
        }
    """
    genre_key = normalize(request.genre) if request.genre and request.genre.strip() else None
    artist_key = normalize(request.artist) if request.artist and request.artist.strip() else None
    if genre_key is None and artist_key is None:
        raise HTTPException(status_code=400, detail={"message": "Informe 'genre' e/ou 'artist'."})

    by_genre = songs["genre_key"] == genre_key
    by_artist = songs["artist_key"] == artist_key
    warnings = []
    if genre_key is not None and not by_genre.any():
        warnings.append({"message": f"Gênero '{request.genre}' não encontrado.",
                         "suggestions": suggest(request.genre, df["genre"])})
    if artist_key is not None and not by_artist.any():
        warnings.append({"message": f"Artista '{request.artist}' não encontrado.",
                         "suggestions": suggest(request.artist, df["artist"])})

    matches = songs[by_genre | by_artist].copy()
    if matches.empty:
        raise HTTPException(status_code=404, detail={
            "message": "Nenhuma música encontrada para o gênero/artista informado.",
            "details": warnings,
        })
    matches["match"] = np.select(
        [by_genre[matches.index] & by_artist[matches.index], by_genre[matches.index]],
        ["genre+artist", "genre"],
        default="artist",
    )
    top = matches.sort_values("Popularity", ascending=False, kind="mergesort").head(request.limit)

    response = {
        "genre": request.genre,
        "artist": request.artist,
        "total_matches": len(matches),
        "total": len(top),
        "recommendations": [{**song_info(row), "match": row["match"]} for _, row in top.iterrows()],
    }
    if warnings:
        response["warnings"] = warnings
    return response


@app.get("/recommendations/collaborative/{user_id}", tags=["Recomendações"],
         summary="3. Filtro colaborativo")
async def collaborative_recommendations(
    user_id: str,
    limit: int = Query(5, ge=1, le=100, description="Número de recomendações a retornar"),
):
    """
    Recomendações baseadas no comportamento de **usuários similares**, com dados fictícios
    de "quem gostou disso também gostou daquilo" (usuários `user1` a `user15`, veja `GET /users`).

    Lógica de **contagem de co-ocorrências**: para cada par de músicas contamos quantos
    usuários gostaram das duas. O `score` de uma música que o usuário ainda não curtiu é a
    soma das co-ocorrências dela com as músicas que ele curtiu. Empates são decididos pela
    popularidade. `because_you_liked` mostra quais músicas curtidas geraram a recomendação.

    **Exemplo de requisição**

        GET /recommendations/collaborative/user1?limit=3

    **Exemplo de resposta** (`liked_songs` resumido)

        {
          "user_id": "user1",
          "liked_songs": [{"title": "Roar", "artist": "Katy Perry"}, "..."],
          "similar_users": [{"user_id": "user2", "songs_in_common": 3},
                            {"user_id": "user3", "songs_in_common": 2}],
          "total": 3,
          "recommendations": [
            {"title": "Die Young", "artist": "Kesha", "genre": "dance pop", "year": 2013,
             "popularity": 75, "score": 5, "because_you_liked": ["Roar", "Dark Horse", "Bad Romance", "TiK ToK", "Telephone"]},
            {"title": "Part Of Me", "artist": "Katy Perry", "genre": "dance pop", "year": 2012,
             "popularity": 71, "score": 3, "because_you_liked": ["Roar", "Bad Romance", "Telephone"]},
            {"title": "Diamonds", "artist": "Rihanna", "genre": "barbadian pop", "year": 2012,
             "popularity": 61, "score": 3, "because_you_liked": ["Roar", "Bad Romance", "Telephone"]}
          ]
        }
    """
    user = resolve_user(user_id)
    liked = set(user_song_keys[user])
    scores, reasons = collaborative_scores(user)
    ranked = sorted(scores, key=lambda key: (-scores[key], -int(df.at[canonical_index[key], "Popularity"])))

    similar_users = []
    for other, keys in user_song_keys.items():
        common = len(liked & set(keys))
        if other != user and common > 0:
            similar_users.append({"user_id": other, "songs_in_common": common})
    similar_users.sort(key=lambda item: -item["songs_in_common"])

    recommendations = []
    for key in ranked[:limit]:
        row = df.loc[canonical_index[key]]
        recommendations.append({
            **song_info(row),
            "score": int(scores[key]),
            "because_you_liked": [title_of(liked_key) for liked_key in reasons[key]],
        })

    return {
        "user_id": user,
        "liked_songs": [short_info(key) for key in user_song_keys[user]],
        "similar_users": similar_users,
        "total": len(recommendations),
        "recommendations": recommendations,
    }


@app.post("/recommendations/hybrid", tags=["Recomendações"], summary="4. Recomendação híbrida")
async def hybrid_recommendations(request: HybridRequest):
    """
    Combina a recomendação **baseada em conteúdo** (similaridade com `song_title`) com o
    **filtro colaborativo** (co-ocorrências do `user_id`):

        hybrid_score = content_weight * content_score + collab_weight * collab_score

    - Os dois scores são normalizados entre 0 e 1 antes da combinação.
    - Os pesos são normalizados para somar 1 (ex.: 2 e 1 viram 0.667 e 0.333).
    - Não recomenda a própria música nem músicas que o usuário já curtiu.

    **Exemplo de requisição**

        POST /recommendations/hybrid
        {"song_title": "Shape of You", "user_id": "user12", "content_weight": 0.7,
         "collab_weight": 0.3, "limit": 5}

    **Exemplo de resposta**

        {
          "song": {"title": "Shape of You", "artist": "Ed Sheeran", "genre": "pop", "year": 2017,
                   "popularity": 87},
          "user_id": "user12",
          "weights": {"content_weight": 0.7, "collab_weight": 0.3},
          "total": 5,
          "recommendations": [
            {"title": "I'm Not The Only One", "artist": "Sam Smith", "genre": "pop", "year": 2015,
             "popularity": 84, "content_score": 0.9704, "collab_score": 1.0, "hybrid_score": 0.9793},
            {"title": "Let Her Go", "artist": "Passenger", "genre": "folk-pop", "year": 2014,
             "popularity": 77, "content_score": 0.8204, "collab_score": 1.0, "hybrid_score": 0.8742},
            {"title": "All I Ask", "artist": "Adele", "genre": "british soul", "year": 2016,
             "popularity": 71, "content_score": 0.7159, "collab_score": 1.0, "hybrid_score": 0.8012},
            {"title": "Closer", "artist": "The Chainsmokers", "genre": "electropop", "year": 2017,
             "popularity": 86, "content_score": 1.0, "collab_score": 0.0, "hybrid_score": 0.7},
            {"title": "Rockabye (feat. Sean Paul & Anne-Marie)", "artist": "Clean Bandit",
             "genre": "dance pop", "year": 2017, "popularity": 78, "content_score": 0.9943,
             "collab_score": 0.0, "hybrid_score": 0.696}
          ]
        }
    """
    index = find_song_index(request.song_title)
    user = resolve_user(request.user_id)
    total_weight = request.content_weight + request.collab_weight
    if total_weight <= 0:
        raise HTTPException(status_code=400, detail={
            "message": "content_weight + collab_weight deve ser maior que zero.",
        })
    content_weight = request.content_weight / total_weight
    collab_weight = request.collab_weight / total_weight

    seed = df.loc[index]
    excluded = set(user_song_keys[user]) | {seed["song_key"]}
    candidates = songs[~songs["song_key"].isin(excluded)].copy()

    content = similarity_matrix[index][candidates.index]
    content_range = content.max() - content.min()
    content_score = (content - content.min()) / content_range if content_range > 0 else np.zeros(len(content))

    scores, _ = collaborative_scores(user)
    collab = np.array([scores.get(key, 0) for key in candidates["song_key"]], dtype=float)
    collab_score = collab / collab.max() if collab.max() > 0 else np.zeros(len(collab))

    candidates["content_score"] = content_score
    candidates["collab_score"] = collab_score
    candidates["hybrid_score"] = content_weight * content_score + collab_weight * collab_score
    top = candidates.sort_values(["hybrid_score", "Popularity"], ascending=False, kind="mergesort").head(request.limit)

    return {
        "song": song_info(seed),
        "user_id": user,
        "weights": {"content_weight": round(content_weight, 4), "collab_weight": round(collab_weight, 4)},
        "total": len(top),
        "recommendations": [
            {
                **song_info(row),
                "content_score": round(float(row["content_score"]), 4),
                "collab_score": round(float(row["collab_score"]), 4),
                "hybrid_score": round(float(row["hybrid_score"]), 4),
            }
            for _, row in top.iterrows()
        ],
    }


@app.get("/recommendations/popular", tags=["Recomendações"], summary="5. Recomendação por popularidade/ano")
async def popular_recommendations(
    year: Optional[int] = Query(None, description="Filtrar por ano (2010 a 2019)"),
    genre: Optional[str] = Query(None, description="Filtrar por gênero (ex.: 'dance pop')"),
    limit: int = Query(5, ge=1, le=100, description="Número de resultados"),
):
    """
    Retorna as músicas mais populares, opcionalmente filtradas por `year` e/ou `genre`,
    ordenadas por popularidade (maior primeiro). Sem filtros, considera o dataset inteiro.

    **Exemplo de requisição**

        GET /recommendations/popular?year=2015&genre=dance%20pop&limit=3

    **Exemplo de resposta**

        {
          "filters": {"year": 2015, "genre": "dance pop"},
          "total": 3,
          "recommendations": [
            {"title": "Uptown Funk", "artist": "Mark Ronson", "genre": "dance pop", "year": 2015,
             "popularity": 82},
            {"title": "Love Me Like You Do - From \\"Fifty Shades Of Grey\\"", "artist": "Ellie Goulding",
             "genre": "dance pop", "year": 2015, "popularity": 79},
            {"title": "Want to Want Me", "artist": "Jason Derulo", "genre": "dance pop", "year": 2015,
             "popularity": 77}
          ]
        }
    """
    data = df
    if year is not None:
        data = data[data["year"] == year]
        if data.empty:
            raise HTTPException(status_code=404, detail={
                "message": f"Não há músicas do ano {year} no dataset.",
                "available_years": sorted(int(y) for y in df["year"].unique()),
            })
    if genre and genre.strip():
        genre_key = normalize(genre)
        if not (df["genre_key"] == genre_key).any():
            raise HTTPException(status_code=404, detail={
                "message": f"Gênero '{genre}' não encontrado.",
                "suggestions": suggest(genre, df["genre"]),
            })
        data = data[data["genre_key"] == genre_key]
        if data.empty:
            raise HTTPException(status_code=404, detail={
                "message": f"Nenhuma música do gênero '{genre}' no ano {year}.",
            })

    top = (data.sort_values("Popularity", ascending=False, kind="mergesort")
               .drop_duplicates("song_key")
               .head(limit))
    return {
        "filters": {"year": year, "genre": genre},
        "total": len(top),
        "recommendations": [song_info(row) for _, row in top.iterrows()],
    }

Writing main.py


### Conferindo o pré-processamento

In [5]:
import importlib

import main
importlib.reload(main)  # garante que a versão mais recente do main.py está carregada

print(f"Linhas após a limpeza: {len(main.df)} | músicas únicas: {len(main.songs)}")
print("Características usadas na similaridade:", main.features)
main.df[["title", "artist", "genre", "year"] + main.features].head()

Linhas após a limpeza: 602 | músicas únicas: 586
Características usadas na similaridade: ['Beats.Per.Minute', 'Energy', 'Danceability', 'Loudness/dB', 'Liveness', 'Valence', 'Length', 'Acousticness', 'Speechiness', 'Popularity']


,title,artist,genre,year,Beats.Per.Minute,Energy,Danceability,Loudness/dB,Liveness,Valence,Length,Acousticness,Speechiness,Popularity
0,"Hey, Soul Sister",Train,neo mellow,2010,97,89,67,-4,8,80,217,19,4,83
1,Love The Way You Lie,Eminem,detroit hip hop,2010,87,93,75,-5,52,64,263,24,23,82
2,TiK ToK,Kesha,dance pop,2010,120,84,76,-3,29,71,200,10,14,80
3,Bad Romance,Lady Gaga,dance pop,2010,119,92,70,-4,8,71,295,0,4,79
4,Just the Way You Are,Bruno Mars,pop,2010,109,84,64,-5,9,43,221,2,4,78


In [6]:
# Títulos com caracteres corrompidos no CSV e a correção aplicada
corrigidos = main.df.loc[main.df["title"] != main.df["original_title"], ["original_title", "title"]]
corrigidos.rename(columns={"original_title": "No CSV", "title": "Corrigido"})

,No CSV,Corrigido
70,Just Can抰 Get Enough,Just Can’t Get Enough
132,We Are Young (feat. Janelle Mon醗),We Are Young (feat. Janelle Monáe)
192,Atlas - From 揟he Hunger Games: Catching Fire?Soundtrack,Atlas - From “The Hunger Games: Catching Fire?Soundtrack
258,Boom Clap - From the Motion Picture Das Schicksal ist ein mieser Verr鋞er,Boom Clap - From the Motion Picture Das Schicksal ist ein mieser Verräter
422,Wish That You Were Here - From 揗iss Peregrine抯 Home for Peculiar Children?Original Motion Picture,Wish That You Were Here - From “Miss Peregrine’s Home for Peculiar Children?Original Motion Picture
456,I Don抰 Wanna Live Forever (Fifty Shades Darker),I Don’t Wanna Live Forever (Fifty Shades Darker)
471,Reggaet髇 Lento (Remix),Reggaetón Lento (Remix)
484,Bon app閠it,Bon appétit
496,No Vacancy (with Sebasti醤 Yatra),No Vacancy (with Sebastián Yatra)
574,Se駉rita,Señorita


In [7]:
# Dados fictícios do filtro colaborativo: "usuários que gostaram de X"
pd.DataFrame(
    [(usuario, len(musicas), " | ".join(musicas)) for usuario, musicas in main.user_likes.items()],
    columns=["user_id", "curtidas", "músicas curtidas"],
)

,user_id,curtidas,músicas curtidas
0,user1,6,Roar | Dark Horse | Bad Romance | TiK ToK | Only Girl (In The World) | Telephone
1,user2,7,Roar | Bad Romance | Telephone | Die Young | Part Of Me | Diamonds | We Found Love
2,user3,7,Dark Horse | TiK ToK | Die Young | Starships | Problem | Side To Side | Titanium (feat. Sia)
3,user4,7,Just the Way You Are | Locked Out of Heaven | Uptown Funk | Sugar | Payphone | Treasure | That's What I Like
4,user5,7,"Uptown Funk | That's What I Like | 24K Magic | Sugar | Animals | Marry You | Happy - From ""Despicable Me 2"""
5,user6,7,Wake Me Up | Summer | Titanium (feat. Sia) | Clarity | Don't Let Me Down | Closer | Hey Brother
6,user7,6,Wake Me Up | Titanium (feat. Sia) | This Is What You Came For | Closer | Something Just Like This | Stay
7,user8,7,Summer | How Deep Is Your Love | One Kiss (with Dua Lipa) | New Rules | Silence | Happier | Don't Let Me Down
8,user9,7,Love Yourself | Sorry | What Do You Mean? | Stitches | Treat You Better | Starboy | The Hills
9,user10,7,Love Yourself | Sorry | Stitches | Mercy | There's Nothing Holdin' Me Back | Baby | Boyfriend


## 4. Testes automatizados (`pytest`)

Os 23 testes cobrem as 5 APIs: resultados, ordenação, parâmetros (`limit`, `weights`, filtros),
pesos da híbrida, conferência independente da contagem de co-ocorrências e respostas de erro
(400, 404, 422).

In [8]:
%%writefile test_main.py
"""
Testes das 5 APIs de recomendação.

Executar:  python -m pytest -v test_main.py
"""

from urllib.parse import quote

from fastapi.testclient import TestClient

from main import app, df, features, songs, user_likes

client = TestClient(app)


def content_url(title: str) -> str:
    return "/recommendations/content-based/" + quote(title, safe="")


def titles(response) -> list:
    return [item["title"] for item in response.json()["recommendations"]]


# ---------------------------------------------------------------------------
# Dados e pré-processamento
# ---------------------------------------------------------------------------

def test_dataset_preprocessado():
    assert set(features + ["title", "artist", "genre", "year"]).issubset(df.columns)
    assert len(df) == 602                      # 603 linhas - 1 registro inválido (BPM 0)
    assert (df["Beats.Per.Minute"] > 0).all()
    assert "Señorita" in set(df["title"])      # caracteres corrigidos ("Se駉rita")
    assert songs["song_key"].is_unique


def test_raiz_e_usuarios():
    response = client.get("/")
    assert response.status_code == 200
    assert len(response.json()["endpoints"]) == 5

    response = client.get("/users")
    assert response.status_code == 200
    assert set(response.json()["users"]) == set(user_likes)


def test_rotas_do_enunciado():
    paths = client.get("/openapi.json").json()["paths"]
    assert "get" in paths["/recommendations/content-based/{song_title}"]
    assert "post" in paths["/recommendations/genre-artist"]
    assert "get" in paths["/recommendations/collaborative/{user_id}"]
    assert "post" in paths["/recommendations/hybrid"]
    assert "get" in paths["/recommendations/popular"]


# ---------------------------------------------------------------------------
# 1. Baseada em conteúdo
# ---------------------------------------------------------------------------

def test_conteudo_padrao():
    response = client.get(content_url("Hey, Soul Sister"))
    assert response.status_code == 200
    data = response.json()
    assert data["song"]["title"] == "Hey, Soul Sister"
    assert data["total"] == 5                                     # limit padrão
    assert "Hey, Soul Sister" not in titles(response)             # não recomenda a própria música
    similarities = [item["similarity"] for item in data["recommendations"]]
    assert similarities == sorted(similarities, reverse=True)
    assert all(0 <= value <= 1 for value in similarities)


def test_conteudo_limit_e_sem_repeticao():
    response = client.get(content_url("Love Yourself"), params={"limit": 50})
    assert response.status_code == 200
    items = response.json()["recommendations"]
    assert len(items) == 50
    assert len({(item["title"], item["artist"]) for item in items}) == 50


def test_conteudo_weights_json_e_formato_curto():
    sem_pesos = client.get(content_url("Shape of You"), params={"limit": 10})
    com_pesos = client.get(content_url("Shape of You"),
                           params={"limit": 10, "weights": '{"Beats.Per.Minute": 10, "Energy": 0}'})
    assert com_pesos.status_code == 200
    assert com_pesos.json()["weights"]["Beats.Per.Minute"] == 10
    assert com_pesos.json()["weights"]["Energy"] == 0
    assert com_pesos.json()["weights"]["Danceability"] == 1       # não informado -> peso 1
    assert titles(com_pesos) != titles(sem_pesos)                  # os pesos mudam o resultado

    curto = client.get(content_url("Shape of You"), params={"limit": 10, "weights": "bpm:10,energy:0"})
    assert curto.status_code == 200
    assert titles(curto) == titles(com_pesos)


def test_conteudo_weights_invalidos():
    for weights in ['{"Energia": 2}', '{"Energy": -1}', "abc", '{"Energy": "x"}', "[1, 2]",
                    '{"' + '": 0, "'.join(features) + '": 0}']:
        response = client.get(content_url("Shape of You"), params={"weights": weights})
        assert response.status_code == 400, weights


def test_conteudo_busca_flexivel():
    assert client.get(content_url("hey, soul sister")).json()["song"]["title"] == "Hey, Soul Sister"
    assert client.get(content_url("senorita")).json()["song"]["title"] == "Señorita"
    assert client.get(content_url("Titanium")).json()["song"]["title"] == "Titanium (feat. Sia)"
    # título com "/" funciona na URL
    response = client.get(content_url('Let It Go - From "Frozen / Single Version'))
    assert response.status_code == 200


def test_conteudo_titulo_repetido_com_artista():
    assert client.get(content_url("Hello")).json()["song"]["artist"] == "Adele"   # mais popular
    response = client.get(content_url("Hello"), params={"artist": "Martin Solveig"})
    assert response.json()["song"]["artist"] == "Martin Solveig"


def test_conteudo_erros():
    response = client.get(content_url("Musica Que Nao Existe"))
    assert response.status_code == 404
    assert "suggestions" in response.json()["detail"]
    assert client.get(content_url("Shape of You"), params={"limit": 0}).status_code == 422
    assert client.get(content_url("Shape of You"), params={"limit": 101}).status_code == 422


# ---------------------------------------------------------------------------
# 2. Gênero/Artista
# ---------------------------------------------------------------------------

def test_genero():
    response = client.post("/recommendations/genre-artist", json={"genre": "Dance Pop", "limit": 10})
    assert response.status_code == 200
    items = response.json()["recommendations"]
    assert len(items) == 10
    assert all(item["genre"] == "dance pop" and item["match"] == "genre" for item in items)
    popularity = [item["popularity"] for item in items]
    assert popularity == sorted(popularity, reverse=True)


def test_artista():
    response = client.post("/recommendations/genre-artist", json={"artist": "adele", "limit": 50})
    assert response.status_code == 200
    items = response.json()["recommendations"]
    assert items and all(item["artist"] == "Adele" for item in items)
    assert len({item["title"] for item in items}) == len(items)   # sem repetição de anos


def test_genero_ou_artista():
    response = client.post("/recommendations/genre-artist",
                           json={"genre": "canadian pop", "artist": "Ed Sheeran", "limit": 100})
    data = response.json()
    assert response.status_code == 200
    assert {item["match"] for item in data["recommendations"]} == {"genre", "artist"}
    assert all(item["genre"] == "canadian pop" or item["artist"] == "Ed Sheeran"
               for item in data["recommendations"])
    popularity = [item["popularity"] for item in data["recommendations"]]
    assert popularity == sorted(popularity, reverse=True)


def test_genero_artista_erros():
    assert client.post("/recommendations/genre-artist", json={}).status_code == 400
    assert client.post("/recommendations/genre-artist", json={"genre": "xyz", "artist": "xyz"}).status_code == 404
    assert client.post("/recommendations/genre-artist", json={"genre": "pop", "limit": 0}).status_code == 422
    # um filtro válido e outro inexistente: devolve o válido e avisa
    response = client.post("/recommendations/genre-artist", json={"genre": "xyz", "artist": "Adele"})
    assert response.status_code == 200
    assert "warnings" in response.json()


# ---------------------------------------------------------------------------
# 3. Filtro colaborativo
# ---------------------------------------------------------------------------

def test_colaborativo():
    response = client.get("/recommendations/collaborative/user1")
    assert response.status_code == 200
    data = response.json()
    liked = {item["title"] for item in data["liked_songs"]}
    assert data["total"] == 5
    assert not liked & set(titles(response))                     # não recomenda o que já curtiu
    scores = [item["score"] for item in data["recommendations"]]
    assert scores == sorted(scores, reverse=True)
    assert data["similar_users"][0] == {"user_id": "user2", "songs_in_common": 3}


def test_colaborativo_contagem_de_coocorrencias():
    # Conferência independente: score(m) = soma, para cada outro usuário que curtiu m,
    # do número de músicas em comum com o usuário.
    liked = set(user_likes["user1"])
    expected = {}
    for other, other_likes in user_likes.items():
        if other == "user1":
            continue
        common = len(liked & set(other_likes))
        for title in other_likes:
            if title not in liked and common:
                expected[title] = expected.get(title, 0) + common
    response = client.get("/recommendations/collaborative/user1", params={"limit": 100})
    got = {item["title"]: item["score"] for item in response.json()["recommendations"]}
    assert got == expected


def test_colaborativo_ids_e_erros():
    assert client.get("/recommendations/collaborative/1").json()["user_id"] == "user1"
    assert client.get("/recommendations/collaborative/USER2").json()["user_id"] == "user2"
    response = client.get("/recommendations/collaborative/user999")
    assert response.status_code == 404
    assert "available_users" in response.json()["detail"]


# ---------------------------------------------------------------------------
# 4. Híbrida
# ---------------------------------------------------------------------------

def hybrid(**body):
    payload = {"song_title": "Shape of You", "user_id": "user12"}
    payload.update(body)
    return client.post("/recommendations/hybrid", json=payload)


def test_hibrida_padrao():
    response = hybrid()
    assert response.status_code == 200
    data = response.json()
    assert data["weights"] == {"content_weight": 0.7, "collab_weight": 0.3}
    assert data["total"] == 5
    liked = set(user_likes["user12"])
    for item in data["recommendations"]:
        assert item["title"] != "Shape of You" and item["title"] not in liked
        expected = 0.7 * item["content_score"] + 0.3 * item["collab_score"]
        assert abs(item["hybrid_score"] - expected) < 1e-3
    scores = [item["hybrid_score"] for item in data["recommendations"]]
    assert scores == sorted(scores, reverse=True)
    # mistura as duas abordagens: há músicas vindas do colaborativo e só do conteúdo
    assert any(item["collab_score"] > 0 for item in data["recommendations"])
    assert any(item["collab_score"] == 0 for item in data["recommendations"])


def test_hibrida_pesos_extremos():
    so_conteudo = hybrid(content_weight=1, collab_weight=0, limit=10)
    conteudo = client.get(content_url("Shape of You"), params={"limit": 20})
    liked = set(user_likes["user12"])
    esperado = [title for title in titles(conteudo) if title not in liked][:10]
    assert titles(so_conteudo) == esperado

    so_colab = hybrid(content_weight=0, collab_weight=1, limit=3)
    assert all(item["collab_score"] == 1.0 for item in so_colab.json()["recommendations"])

    # pesos são normalizados para somar 1
    assert hybrid(content_weight=2, collab_weight=1).json()["weights"] == {
        "content_weight": 0.6667, "collab_weight": 0.3333}


def test_hibrida_erros():
    assert hybrid(content_weight=0, collab_weight=0).status_code == 400
    assert hybrid(content_weight=-1).status_code == 422
    assert hybrid(song_title="Musica Que Nao Existe").status_code == 404
    assert hybrid(user_id="user999").status_code == 404
    assert client.post("/recommendations/hybrid", json={"song_title": "Roar"}).status_code == 422


# ---------------------------------------------------------------------------
# 5. Popularidade/Ano
# ---------------------------------------------------------------------------

def test_popular_sem_filtros():
    response = client.get("/recommendations/popular")
    assert response.status_code == 200
    items = response.json()["recommendations"]
    assert len(items) == 5
    assert items[0]["popularity"] == int(df["Popularity"].max())
    popularity = [item["popularity"] for item in items]
    assert popularity == sorted(popularity, reverse=True)


def test_popular_filtros():
    response = client.get("/recommendations/popular", params={"year": 2015, "genre": "dance pop", "limit": 10})
    assert response.status_code == 200
    items = response.json()["recommendations"]
    assert len(items) == 10
    assert all(item["year"] == 2015 and item["genre"] == "dance pop" for item in items)

    response = client.get("/recommendations/popular", params={"year": 2016, "limit": 100})
    items = response.json()["recommendations"]
    assert all(item["year"] == 2016 for item in items)
    assert len({(item["title"], item["artist"]) for item in items}) == len(items)


def test_popular_erros():
    assert client.get("/recommendations/popular", params={"year": 2005}).status_code == 404
    assert client.get("/recommendations/popular", params={"genre": "xyz"}).status_code == 404
    assert client.get("/recommendations/popular", params={"year": 2010, "genre": "latin"}).status_code == 404
    assert client.get("/recommendations/popular", params={"limit": 0}).status_code == 422

Writing test_main.py


In [9]:
import sys

!{sys.executable} -m pytest -v -p no:warnings -p no:cacheprovider test_main.py

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
rootdir: /content
plugins: langsmith-0.12.1, anyio-4.14.2, typeguard-4.6.0
collected 23 items                                                             

test_main.py::test_dataset_preprocessado PASSED                          [  4%]
test_main.py::test_raiz_e_usuarios PASSED                                [  8%]
test_main.py::test_rotas_do_enunciado PASSED                             [ 13%]
test_main.py::test_conteudo_padrao PASSED                                [ 17%]
test_main.py::test_conteudo_limit_e_sem_repeticao PASSED                 [ 21%]
test_main.py::test_conteudo_weights_json_e_formato_curto PASSED          [ 26%]
test_main.py::test_conteudo_weights_invalidos PASSED                     [ 30%]
test_main.py::test_conteudo_busca_flexivel PASSED                        [ 34%]
test_main.py::test_conteudo_titulo_repetid

## 5. Executando a API com `uvicorn` e chamando cada endpoint

A API sobe em segundo plano (porta 8000) e cada endpoint é chamado por HTTP com `requests`.
As saídas abaixo servem de **exemplos de requisição e resposta**.

In [10]:
import subprocess
import sys
import time

import requests

BASE_URL = "http://127.0.0.1:8000"

try:  # ao reexecutar a célula, para o servidor anterior
    servidor.terminate()
    servidor.wait(timeout=10)
except NameError:
    pass

servidor = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=open("uvicorn.log", "w"), stderr=subprocess.STDOUT,
)
for _ in range(60):
    try:
        requests.get(BASE_URL + "/", timeout=1)
        break
    except requests.ConnectionError:
        time.sleep(0.5)
else:
    print(open("uvicorn.log").read())
    raise RuntimeError("A API não iniciou. Veja o log acima.")

print("API rodando em", BASE_URL)

API rodando em http://127.0.0.1:8000


In [11]:
import json
from urllib.parse import quote, unquote


def musica(titulo):
    """Caminho do endpoint 1 com o título codificado para URL."""
    return "/recommendations/content-based/" + quote(titulo, safe="")


def chamar(metodo, caminho, corpo=None, params=None, mostrar_json=True):
    """Faz a requisição HTTP, mostra a requisição e a resposta e devolve o JSON."""
    resposta = requests.request(metodo, BASE_URL + caminho, json=corpo, params=params)
    consulta = "&".join(f"{chave}={valor}" for chave, valor in (params or {}).items())
    print(f"REQUISIÇÃO: {metodo} {unquote(caminho)}" + (f"?{consulta}" if consulta else ""))
    if corpo is not None:
        print("CORPO:", json.dumps(corpo, ensure_ascii=False))
    print("STATUS:", resposta.status_code)
    dados = resposta.json()
    if mostrar_json:
        print("RESPOSTA:")
        print(json.dumps(dados, ensure_ascii=False, indent=2))
    return dados


def tabela(dados):
    """Mostra as recomendações em formato de tabela."""
    resultado = pd.json_normalize(dados["recommendations"])
    resultado = resultado.drop(columns=["features.Popularity"], errors="ignore")
    resultado.columns = [coluna.replace("features.", "") for coluna in resultado.columns]
    resultado.index = range(1, len(resultado) + 1)
    return resultado


chamar("GET", "/");

REQUISIÇÃO: GET /
STATUS: 200
RESPOSTA:
{
  "message": "Sistema de Recomendação Musical",
  "docs": "/docs",
  "dataset": {
    "rows": 602,
    "unique_songs": 586,
    "years": [
      2010,
      2011,
      2012,
      2013,
      2014,
      2015,
      2016,
      2017,
      2018,
      2019
    ],
    "genres": 50,
    "artists": 184
  },
  "endpoints": [
    "GET  /recommendations/content-based/{song_title}",
    "POST /recommendations/genre-artist",
    "GET  /recommendations/collaborative/{user_id}",
    "POST /recommendations/hybrid",
    "GET  /recommendations/popular"
  ]
}


### 5.1 Recomendação baseada em conteúdo

`GET /recommendations/content-based/{song_title}`, com os parâmetros opcionais `limit` (padrão 5) e
`weights` (pesos por característica, em JSON). O retorno são as músicas mais parecidas pela
similaridade do cosseno entre as características normalizadas.

In [12]:
chamar("GET", musica("Shape of You"), params={"limit": 3});

REQUISIÇÃO: GET /recommendations/content-based/Shape of You?limit=3
STATUS: 200
RESPOSTA:
{
  "song": {
    "title": "Shape of You",
    "artist": "Ed Sheeran",
    "genre": "pop",
    "year": 2017,
    "popularity": 87,
    "features": {
      "Beats.Per.Minute": 96,
      "Energy": 65,
      "Danceability": 83,
      "Loudness/dB": -3,
      "Liveness": 9,
      "Valence": 93,
      "Length": 234,
      "Acousticness": 58,
      "Speechiness": 8,
      "Popularity": 87
    }
  },
  "weights": {
    "Beats.Per.Minute": 1.0,
    "Energy": 1.0,
    "Danceability": 1.0,
    "Loudness/dB": 1.0,
    "Liveness": 1.0,
    "Valence": 1.0,
    "Length": 1.0,
    "Acousticness": 1.0,
    "Speechiness": 1.0,
    "Popularity": 1.0
  },
  "total": 3,
  "recommendations": [
    {
      "title": "Closer",
      "artist": "The Chainsmokers",
      "genre": "electropop",
      "year": 2017,
      "popularity": 86,
      "similarity": 0.9878,
      "features": {
        "Beats.Per.Minute": 95,
        

In [13]:
# Com pesos: BPM e energia passam a contar mais na similaridade
pesos = json.dumps({"Beats.Per.Minute": 3, "Energy": 2})
tabela(chamar("GET", musica("Shape of You"), params={"limit": 5, "weights": pesos}, mostrar_json=False))

REQUISIÇÃO: GET /recommendations/content-based/Shape of You?limit=5&weights={"Beats.Per.Minute": 3, "Energy": 2}
STATUS: 200


,title,artist,genre,year,popularity,similarity,Beats.Per.Minute,Energy,Danceability,Loudness/dB,Liveness,Valence,Length,Acousticness,Speechiness
1,Closer,The Chainsmokers,electropop,2017,86,0.9886,95,52,75,-6,11,66,245,41,3
2,Rockabye (feat. Sean Paul & Anne-Marie),Clean Bandit,dance pop,2017,78,0.9840,102,76,72,-4,18,74,251,41,5
3,There's Nothing Holdin' Me Back,Shawn Mendes,canadian pop,2018,84,0.9818,122,81,87,-4,8,97,199,38,6
4,Sing,Ed Sheeran,pop,2015,71,0.9807,120,67,82,-4,6,94,235,30,5
5,I'm Not The Only One,Sam Smith,pop,2015,84,0.9799,82,49,68,-6,8,49,239,53,4


In [14]:
# Busca flexível: sem acento, título sem complemento e título repetido entre artistas
for titulo, extra in [("senorita", {}), ("Titanium", {}), ("Hello", {}), ("Hello", {"artist": "Martin Solveig"})]:
    dados = requests.get(BASE_URL + musica(titulo), params={"limit": 1, **extra}).json()
    print(f"{titulo!r:<12} {str(extra):<30} -> {dados['song']['title']} ({dados['song']['artist']})")

'senorita'   {}                             -> Señorita (Shawn Mendes)
'Titanium'   {}                             -> Titanium (feat. Sia) (David Guetta)
'Hello'      {}                             -> Hello (Adele)
'Hello'      {'artist': 'Martin Solveig'}   -> Hello (Martin Solveig)


In [15]:
# Erros: música inexistente (404 com sugestões) e característica inválida em weights (400)
chamar("GET", musica("Shap of Yuo"))
print()
chamar("GET", musica("Shape of You"), params={"weights": json.dumps({"Energia": 2})});

REQUISIÇÃO: GET /recommendations/content-based/Shap of Yuo
STATUS: 404
RESPOSTA:
{
  "detail": {
    "message": "Música 'Shap of Yuo' não encontrada.",
    "suggestions": [
      "Shape of You",
      "Shake It Off",
      "Hall of Fame",
      "What Do You Mean?",
      "There for You"
    ]
  }
}

REQUISIÇÃO: GET /recommendations/content-based/Shape of You?weights={"Energia": 2}
STATUS: 400
RESPOSTA:
{
  "detail": {
    "message": "Característica desconhecida em weights: 'Energia'.",
    "valid_features": [
      "Beats.Per.Minute",
      "Energy",
      "Danceability",
      "Loudness/dB",
      "Liveness",
      "Valence",
      "Length",
      "Acousticness",
      "Speechiness",
      "Popularity"
    ]
  }
}


### 5.2 Recomendação por gênero/artista

`POST /recommendations/genre-artist`, com corpo `{"genre", "artist", "limit"}`. Retorna músicas do
mesmo gênero **ou** artista, ordenadas por popularidade. O campo `match` indica o motivo.

In [16]:
chamar("POST", "/recommendations/genre-artist", corpo={"genre": "canadian pop", "artist": "Ed Sheeran", "limit": 3});

REQUISIÇÃO: POST /recommendations/genre-artist
CORPO: {"genre": "canadian pop", "artist": "Ed Sheeran", "limit": 3}
STATUS: 200
RESPOSTA:
{
  "genre": "canadian pop",
  "artist": "Ed Sheeran",
  "total_matches": 42,
  "total": 3,
  "recommendations": [
    {
      "title": "Señorita",
      "artist": "Shawn Mendes",
      "genre": "canadian pop",
      "year": 2019,
      "popularity": 95,
      "match": "genre"
    },
    {
      "title": "South of the Border (feat. Camila Cabello & Cardi B)",
      "artist": "Ed Sheeran",
      "genre": "pop",
      "year": 2019,
      "popularity": 92,
      "match": "artist"
    },
    {
      "title": "Shape of You",
      "artist": "Ed Sheeran",
      "genre": "pop",
      "year": 2017,
      "popularity": 87,
      "match": "artist"
    }
  ]
}


In [17]:
tabela(chamar("POST", "/recommendations/genre-artist", corpo={"artist": "Adele", "limit": 10}, mostrar_json=False))

REQUISIÇÃO: POST /recommendations/genre-artist
CORPO: {"artist": "Adele", "limit": 10}
STATUS: 200


,title,artist,genre,year,popularity,match
1,Someone Like You,Adele,british soul,2011,80,artist
2,Rolling in the Deep,Adele,british soul,2011,76,artist
3,Set Fire to the Rain,Adele,british soul,2012,76,artist
4,Hello,Adele,british soul,2015,73,artist
5,All I Ask,Adele,british soul,2016,71,artist
6,Send My Love (To Your New Lover),Adele,british soul,2016,61,artist
7,When We Were Young,Adele,british soul,2016,58,artist
8,Water Under the Bridge,Adele,british soul,2017,49,artist


In [18]:
chamar("POST", "/recommendations/genre-artist", corpo={});  # sem gênero e sem artista -> 400

REQUISIÇÃO: POST /recommendations/genre-artist
CORPO: {}
STATUS: 400
RESPOSTA:
{
  "detail": {
    "message": "Informe 'genre' e/ou 'artist'."
  }
}


### 5.3 Filtro colaborativo

`GET /recommendations/collaborative/{user_id}` recomenda o que **usuários similares** curtiram. O
`score` é a soma das co-ocorrências com as músicas que o usuário já curtiu, e `because_you_liked`
mostra de onde veio cada recomendação.

In [19]:
chamar("GET", "/recommendations/collaborative/user1", params={"limit": 3});

REQUISIÇÃO: GET /recommendations/collaborative/user1?limit=3
STATUS: 200
RESPOSTA:
{
  "user_id": "user1",
  "liked_songs": [
    {
      "title": "Roar",
      "artist": "Katy Perry"
    },
    {
      "title": "Dark Horse",
      "artist": "Katy Perry"
    },
    {
      "title": "Bad Romance",
      "artist": "Lady Gaga"
    },
    {
      "title": "TiK ToK",
      "artist": "Kesha"
    },
    {
      "title": "Only Girl (In The World)",
      "artist": "Rihanna"
    },
    {
      "title": "Telephone",
      "artist": "Lady Gaga"
    }
  ],
  "similar_users": [
    {
      "user_id": "user2",
      "songs_in_common": 3
    },
    {
      "user_id": "user3",
      "songs_in_common": 2
    }
  ],
  "total": 3,
  "recommendations": [
    {
      "title": "Die Young",
      "artist": "Kesha",
      "genre": "dance pop",
      "year": 2013,
      "popularity": 75,
      "score": 5,
      "because_you_liked": [
        "Roar",
        "Dark Horse",
        "Bad Romance",
        "TiK ToK

In [20]:
tabela(chamar("GET", "/recommendations/collaborative/user9", params={"limit": 8}, mostrar_json=False))

REQUISIÇÃO: GET /recommendations/collaborative/user9?limit=8
STATUS: 200


,title,artist,genre,year,popularity,score,because_you_liked
1,There's Nothing Holdin' Me Back,Shawn Mendes,canadian pop,2018,84,3,"[Love Yourself, Sorry, Stitches]"
2,Mercy,Shawn Mendes,canadian pop,2017,81,3,"[Love Yourself, Sorry, Stitches]"
3,I Feel It Coming,The Weeknd,canadian contemporary r&b,2017,81,3,"[Love Yourself, Starboy, The Hills]"
4,Baby,Justin Bieber,canadian pop,2010,77,3,"[Love Yourself, Sorry, Stitches]"
5,Too Good,Drake,canadian hip hop,2016,74,3,"[Love Yourself, Starboy, The Hills]"
6,"Hold On, We're Going Home",Drake,canadian hip hop,2013,73,3,"[Love Yourself, Starboy, The Hills]"
7,Boyfriend,Justin Bieber,canadian pop,2012,71,3,"[Love Yourself, Sorry, Stitches]"


In [21]:
chamar("GET", "/recommendations/collaborative/user999");  # usuário inexistente -> 404

REQUISIÇÃO: GET /recommendations/collaborative/user999
STATUS: 404
RESPOSTA:
{
  "detail": {
    "message": "Usuário 'user999' não encontrado.",
    "available_users": [
      "user1",
      "user2",
      "user3",
      "user4",
      "user5",
      "user6",
      "user7",
      "user8",
      "user9",
      "user10",
      "user11",
      "user12",
      "user13",
      "user14",
      "user15"
    ]
  }
}


### 5.4 Recomendação híbrida

`POST /recommendations/hybrid`, com corpo `{"song_title", "user_id", "content_weight", "collab_weight", "limit"}`.

`hybrid_score = content_weight × content_score + collab_weight × collab_score`

In [22]:
chamar("POST", "/recommendations/hybrid", corpo={
    "song_title": "Shape of You", "user_id": "user12",
    "content_weight": 0.7, "collab_weight": 0.3, "limit": 5,
});

REQUISIÇÃO: POST /recommendations/hybrid
CORPO: {"song_title": "Shape of You", "user_id": "user12", "content_weight": 0.7, "collab_weight": 0.3, "limit": 5}
STATUS: 200
RESPOSTA:
{
  "song": {
    "title": "Shape of You",
    "artist": "Ed Sheeran",
    "genre": "pop",
    "year": 2017,
    "popularity": 87
  },
  "user_id": "user12",
  "weights": {
    "content_weight": 0.7,
    "collab_weight": 0.3
  },
  "total": 5,
  "recommendations": [
    {
      "title": "I'm Not The Only One",
      "artist": "Sam Smith",
      "genre": "pop",
      "year": 2015,
      "popularity": 84,
      "content_score": 0.9704,
      "collab_score": 1.0,
      "hybrid_score": 0.9793
    },
    {
      "title": "Let Her Go",
      "artist": "Passenger",
      "genre": "folk-pop",
      "year": 2014,
      "popularity": 77,
      "content_score": 0.8204,
      "collab_score": 1.0,
      "hybrid_score": 0.8742
    },
    {
      "title": "All I Ask",
      "artist": "Adele",
      "genre": "british soul",
 

In [23]:
# Efeito dos pesos: 1/0 = só conteúdo (mesma ordem da API 1); 0/1 = só colaborativo (mesma ordem
# da API 3; depois das músicas com score colaborativo, o empate é decidido pela popularidade)
comparacao = {}
for conteudo, colaborativo in [(1.0, 0.0), (0.7, 0.3), (0.5, 0.5), (0.0, 1.0)]:
    dados = requests.post(BASE_URL + "/recommendations/hybrid", json={
        "song_title": "Shape of You", "user_id": "user12",
        "content_weight": conteudo, "collab_weight": colaborativo, "limit": 5,
    }).json()
    comparacao[f"conteúdo {conteudo} / colab {colaborativo}"] = [item["title"] for item in dados["recommendations"]]
pd.DataFrame(comparacao, index=range(1, 6))

,conteúdo 1.0 / colab 0.0,conteúdo 0.7 / colab 0.3,conteúdo 0.5 / colab 0.5,conteúdo 0.0 / colab 1.0
1,Closer,I'm Not The Only One,I'm Not The Only One,I'm Not The Only One
2,Rockabye (feat. Sean Paul & Anne-Marie),Let Her Go,Let Her Go,Let Her Go
3,There's Nothing Holdin' Me Back,All I Ask,All I Ask,All I Ask
4,Sing,Closer,Closer,Memories
5,First Time,Rockabye (feat. Sean Paul & Anne-Marie),Rockabye (feat. Sean Paul & Anne-Marie),Lose You To Love Me


In [24]:
chamar("POST", "/recommendations/hybrid", corpo={
    "song_title": "Shape of You", "user_id": "user12", "content_weight": 0, "collab_weight": 0,
});  # pesos zerados -> 400

REQUISIÇÃO: POST /recommendations/hybrid
CORPO: {"song_title": "Shape of You", "user_id": "user12", "content_weight": 0, "collab_weight": 0}
STATUS: 400
RESPOSTA:
{
  "detail": {
    "message": "content_weight + collab_weight deve ser maior que zero."
  }
}


### 5.5 Recomendação por popularidade/ano

`GET /recommendations/popular`, com os parâmetros opcionais `year`, `genre` e `limit`.

In [25]:
chamar("GET", "/recommendations/popular", params={"year": 2015, "genre": "dance pop", "limit": 3});

REQUISIÇÃO: GET /recommendations/popular?year=2015&genre=dance pop&limit=3
STATUS: 200
RESPOSTA:
{
  "filters": {
    "year": 2015,
    "genre": "dance pop"
  },
  "total": 3,
  "recommendations": [
    {
      "title": "Uptown Funk",
      "artist": "Mark Ronson",
      "genre": "dance pop",
      "year": 2015,
      "popularity": 82
    },
    {
      "title": "Love Me Like You Do - From \"Fifty Shades Of Grey\"",
      "artist": "Ellie Goulding",
      "genre": "dance pop",
      "year": 2015,
      "popularity": 79
    },
    {
      "title": "Want to Want Me",
      "artist": "Jason Derulo",
      "genre": "dance pop",
      "year": 2015,
      "popularity": 77
    }
  ]
}


In [26]:
tabela(chamar("GET", "/recommendations/popular", params={"limit": 10}, mostrar_json=False))

REQUISIÇÃO: GET /recommendations/popular?limit=10
STATUS: 200


,title,artist,genre,year,popularity
1,Memories,Maroon 5,pop,2019,99
2,Lose You To Love Me,Selena Gomez,dance pop,2019,97
3,Someone You Loved,Lewis Capaldi,pop,2019,96
4,Señorita,Shawn Mendes,canadian pop,2019,95
5,How Do You Sleep?,Sam Smith,pop,2019,93
6,South of the Border (feat. Camila Cabello & Cardi B),Ed Sheeran,pop,2019,92
7,Trampoline (with ZAYN),SHAED,electropop,2019,92
8,Happier,Marshmello,brostep,2019,90
9,Truth Hurts,Lizzo,escape room,2019,90
10,Good as Hell (feat. Ariana Grande) - Remix,Lizzo,escape room,2019,90


In [27]:
tabela(chamar("GET", "/recommendations/popular", params={"year": 2019, "limit": 5}, mostrar_json=False))

REQUISIÇÃO: GET /recommendations/popular?year=2019&limit=5
STATUS: 200


,title,artist,genre,year,popularity
1,Memories,Maroon 5,pop,2019,99
2,Lose You To Love Me,Selena Gomez,dance pop,2019,97
3,Someone You Loved,Lewis Capaldi,pop,2019,96
4,Señorita,Shawn Mendes,canadian pop,2019,95
5,How Do You Sleep?,Sam Smith,pop,2019,93


In [28]:
chamar("GET", "/recommendations/popular", params={"year": 2005});  # ano fora do dataset -> 404

REQUISIÇÃO: GET /recommendations/popular?year=2005
STATUS: 404
RESPOSTA:
{
  "detail": {
    "message": "Não há músicas do ano 2005 no dataset.",
    "available_years": [
      2010,
      2011,
      2012,
      2013,
      2014,
      2015,
      2016,
      2017,
      2018,
      2019
    ]
  }
}


## 6. Documentação interativa (Swagger)

O FastAPI gera a documentação automaticamente em `/docs`, com a descrição e os exemplos de cada
endpoint. Pelo botão **Try it out** é possível testar as APIs no navegador.

In [29]:
try:
    from google.colab import output
    print("Clique no link abaixo para abrir o Swagger da API:")
    output.serve_kernel_port_as_window(8000, path="/docs")
except ImportError:
    print(f"Abra {BASE_URL}/docs no navegador.")

Clique no link abaixo para abrir o Swagger da API:
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

## 7. Checklist do enunciado

| Requisito | Status | Onde |
|-----------|:------:|------|
| 5 endpoints com as rotas e métodos pedidos | ✅ | Seção 3 (`main.py`) e teste `test_rotas_do_enunciado` |
| 1. Conteúdo: similaridade por características, `limit` (padrão 5) e `weights` | ✅ | Seção 5.1 |
| 2. Gênero/Artista: POST com `genre`, `artist`, `limit`, ordenado por popularidade | ✅ | Seção 5.2 |
| 3. Colaborativo: dados fictícios + contagem de co-ocorrências | ✅ | Seções 3 e 5.3 |
| 4. Híbrida: POST com `song_title`, `user_id`, `content_weight` (0.7), `collab_weight` (0.3), `limit` | ✅ | Seção 5.4 |
| 5. Popularidade/Ano: GET com `year`, `genre`, `limit` opcionais | ✅ | Seção 5.5 |
| Estrutura do código base (FastAPI, Pydantic, `MinMaxScaler`, `cosine_similarity`) | ✅ | Seção 3 |
| Simular dados de interação usuário-música | ✅ | Seção 3 (`user_likes`) |
| Documentar cada endpoint com exemplos de requisição/resposta | ✅ | Seção 5 e Swagger (seção 6) |
| Testar todas as APIs localmente | ✅ | Seção 4 (23 testes) e seção 5 (servidor real) |

Para parar a API, remova o `#` da célula abaixo e execute-a.

In [30]:
# servidor.terminate()  # para o servidor uvicorn